# MIMIC Property Graph Pipeline: Pre-UMLS then UMLS

This notebook runs one configurable experiment in two stages:

1. `pre_umls`: fast graph construction without UMLS normalization.
2. `umls`: final graph construction with UMLS entity standardization and concept hints.

Set provider, model sweep, note limits, and UMLS controls through environment variables or the setup cell. Use `all`, `none`, or `unlimited` for no cap on note count or note length.

In [ ]:
from __future__ import annotations

import asyncio
import os
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import Image, IFrame, display

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

from eval.medqa_smoke import load_questions, format_options, extract_answer
from helpers.config import env_bool, env_optional_int
from ingest.mimic import MimicDischargeSubsetConfig, extract_mimic_discharge_subset
from rag.index import ensure_index
from rag.retrieve import query_index_context
from rag.visualize import save_clinical_entity_graph, save_clinical_entity_graph_jpeg

experiment_name = os.environ.get("EXPERIMENT_NAME", "mimic_umls_pipeline")
provider = os.environ.get("INDEX_LLM_PROVIDER", os.environ.get("EXPERIMENT_PROVIDER", "vllm")).strip().lower()
embedding_provider = os.environ.get("INDEX_EMBEDDING_PROVIDER", provider).strip().lower()

os.environ["INDEX_LLM_PROVIDER"] = provider
os.environ["INDEX_EMBEDDING_PROVIDER"] = embedding_provider
os.environ.setdefault("INDEX_LLM_REQUEST_TIMEOUT", "240")
os.environ.setdefault("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
os.environ.setdefault("VLLM_API_KEY", "EMPTY")
os.environ.setdefault("OLLAMA_ENDPOINT", "http://127.0.0.1:11434")
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / "output" / ".matplotlib"))

def env_csv(name: str) -> list[str]:
    value = os.environ.get(name, "").strip()
    return [item.strip() for item in value.split(",") if item.strip()]

def model_slug(value: object | None) -> str:
    text = "unknown" if value is None else str(value)
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in text).strip("_") or "unknown"

def combo_slug(generation_model: str, embedding_model: str) -> str:
    return f"gen-{model_slug(generation_model)}__embed-{model_slug(embedding_model)}"

def served_vllm_models(base_url: str) -> list[str]:
    try:
        response = requests.get(f"{base_url.rstrip('/')}/models", timeout=10)
        response.raise_for_status()
    except Exception as exc:
        print(f"Could not read vLLM /models endpoint: {exc}")
        return []
    payload = response.json()
    return [item.get("id") for item in payload.get("data", []) if item.get("id")]

default_generation_models_by_provider = {
    "vllm": [
        "Qwen/Qwen2.5-7B-Instruct",
        "Qwen/Qwen3-8B",
        "mistralai/Mistral-7B-Instruct-v0.3",
        "meta-llama/Llama-3.1-8B-Instruct",
        "google/gemma-2-9b-it",
        "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    ],
    "ollama": ["qwen3.5:0.8b", "qwen3:1.7b", "llama3.2:1b", "gemma3:1b", "mistral"],
    "openai": [os.environ.get("INDEX_LLM_MODEL", "gpt-4.1-mini")],
}
default_embedding_models_by_provider = {
    "vllm": [os.environ.get("VLLM_EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")],
    "ollama": ["embeddinggemma:300m"],
    "openai": [os.environ.get("INDEX_EMBEDDING_MODEL", "text-embedding-3-small")],
}

generation_model_sweep = env_csv("MODEL_SWEEP") or env_csv(f"{provider.upper()}_MODEL_SWEEP") or default_generation_models_by_provider.get(provider, [])
embedding_model_sweep = env_csv("EMBEDDING_MODEL_SWEEP") or env_csv(f"{embedding_provider.upper()}_EMBEDDING_MODEL_SWEEP") or default_embedding_models_by_provider.get(embedding_provider, [])

served_models = []
if provider == "vllm":
    served_models = served_vllm_models(os.environ["VLLM_BASE_URL"])
    if served_models:
        generation_model_sweep = [model for model in generation_model_sweep if model in served_models] or [served_models[0]]

if not generation_model_sweep:
    raise ValueError(f"No generation models configured for provider {provider!r}.")
if not embedding_model_sweep:
    raise ValueError(f"No embedding models configured for provider {embedding_provider!r}.")

print("repo_root:", repo_root)
print("experiment_name:", experiment_name)
print("provider:", provider)
print("embedding_provider:", embedding_provider)
print("served_vllm_models:", served_models or "unknown/not-vllm")
print("generation_model_sweep:", generation_model_sweep)
print("embedding_model_sweep:", embedding_model_sweep)
print("total_model_combinations:", len(generation_model_sweep) * len(embedding_model_sweep))


In [ ]:
mimic_csv = Path(os.environ.get("MIMIC_DISCHARGE_CSV", repo_root / "data" / "mimic_iv_note" / "discharge.csv"))
mimic_notes_dir = Path(os.environ.get("EXPERIMENT_INPUT_DIR", repo_root / "data" / "evidence" / experiment_name))
output_root = Path(os.environ.get("EXPERIMENT_OUTPUT_DIR", repo_root / "output" / experiment_name))
output_root.mkdir(parents=True, exist_ok=True)

note_limit = env_optional_int("MIMIC_DISCHARGE_LIMIT", None)
note_max_chars = env_optional_int("MIMIC_DISCHARGE_MAX_CHARS", 3000)
note_type = os.environ.get("MIMIC_DISCHARGE_NOTE_TYPE", "DS")

test_jsonl_candidates = [
    repo_root / "test.jsonl",
    repo_root / "data" / "medqa" / "data_clean" / "questions" / "US" / "test.jsonl",
    repo_root / "data" / "eval" / "test.jsonl",
]
test_jsonl = next((path for path in test_jsonl_candidates if path.exists()), None)

print("mimic_csv:", mimic_csv)
print("mimic_notes_dir:", mimic_notes_dir)
print("output_root:", output_root)
print("note_limit:", note_limit if note_limit is not None else "all")
print("note_max_chars:", note_max_chars if note_max_chars is not None else "all")
print("note_type:", note_type)
print("test_jsonl:", test_jsonl)


In [ ]:
if not mimic_csv.exists():
    raise FileNotFoundError(f"Missing MIMIC discharge CSV: {mimic_csv}")

written_notes = extract_mimic_discharge_subset(
    MimicDischargeSubsetConfig(
        csv_path=mimic_csv,
        output_dir=mimic_notes_dir,
        limit=note_limit,
        note_type=note_type,
        max_chars=note_max_chars,
        overwrite=env_bool("EXPERIMENT_OVERWRITE_NOTES", True),
    )
)
print(f"Prepared {len(written_notes)} notes in: {mimic_notes_dir}")


In [ ]:
questions = []
if test_jsonl:
    questions = load_questions(test_jsonl, sample_size=int(os.environ.get("EVAL_SAMPLE_SIZE", "1")))
print("question_count:", len(questions))
pd.DataFrame([{"id": q.get("id"), "question": q.get("question"), "answer": q.get("answer")} for q in questions])


In [ ]:
def build_query_llm(model_name: str):
    timeout = float(os.environ.get("INDEX_LLM_REQUEST_TIMEOUT", "240"))
    if provider == "vllm":
        from llama_index.llms.openai import OpenAI

        return OpenAI(
            model=model_name,
            api_key=os.environ.get("VLLM_API_KEY", "EMPTY"),
            api_base=os.environ.get("VLLM_BASE_URL", "http://127.0.0.1:8000/v1"),
            timeout=timeout,
        )
    if provider == "ollama":
        from llama_index.llms.ollama import Ollama

        return Ollama(model=model_name, base_url=os.environ.get("OLLAMA_ENDPOINT", "http://127.0.0.1:11434"), request_timeout=timeout)
    if provider == "openai":
        from llama_index.llms.openai import OpenAI

        return OpenAI(model=model_name, api_key=os.environ.get("OPENAI_API_KEY"), timeout=timeout)
    raise ValueError(f"Unsupported query provider: {provider}")

stages = [
    {"name": "pre_umls", "use_umls": False, "schema_guided": env_bool("PRE_UMLS_SCHEMA_GUIDED", False), "enabled": env_bool("RUN_PRE_UMLS", True)},
    {"name": "umls", "use_umls": True, "schema_guided": env_bool("UMLS_SCHEMA_GUIDED", False), "enabled": env_bool("RUN_UMLS", True)},
]

rows = []
artifact_rows = []

for stage in stages:
    if not stage["enabled"]:
        continue
    os.environ["UMLS_ENABLED"] = "true" if stage["use_umls"] else "false"
    stage_root = output_root / stage["name"]
    artifact_dir = stage_root / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)

    for generation_model in generation_model_sweep:
        os.environ["INDEX_LLM_MODEL"] = generation_model
        if provider == "vllm":
            os.environ["VLLM_MODEL"] = generation_model

        for embedding_model in embedding_model_sweep:
            os.environ["INDEX_EMBEDDING_MODEL"] = embedding_model
            if embedding_provider == "vllm":
                os.environ["VLLM_EMBEDDING_MODEL"] = embedding_model

            slug = combo_slug(generation_model, embedding_model)
            index_dir = stage_root / "indexes" / slug
            html_path = artifact_dir / f"{slug}.html"
            jpeg_path = artifact_dir / f"{slug}.jpg"
            results_path = artifact_dir / f"{slug}_results.csv"

            started = time.time()
            indexed_graph = await asyncio.to_thread(
                ensure_index,
                input_dir=mimic_notes_dir,
                output_dir=index_dir,
                use_umls=stage["use_umls"],
                schema_guided=stage["schema_guided"],
            )
            build_seconds = time.time() - started

            graph_html = save_clinical_entity_graph(index_dir, html_path, source="auto")
            graph_jpeg = save_clinical_entity_graph_jpeg(index_dir, jpeg_path, source="auto", title=f"{stage['name']}: {slug}")
            query_llm = build_query_llm(generation_model)

            combo_rows = []
            for item in questions:
                prompt = item["question"]
                if item.get("options"):
                    prompt = f"{prompt}\n\n" + format_options(item["options"])
                response, context = await query_index_context(index=indexed_graph, query=prompt, llm=query_llm)
                predicted = extract_answer(response, item["options"]) if item.get("options") else None
                combo_rows.append({
                    "stage": stage["name"],
                    "id": item.get("id"),
                    "question": item["question"],
                    "gold_answer": item.get("answer"),
                    "predicted": predicted,
                    "model": generation_model,
                    "embedding_model": embedding_model,
                    "combo_slug": slug,
                    "response": response[:1000],
                    "context_preview": context[:1000] if context else None,
                    "html_plot": graph_html.as_posix(),
                    "jpeg_plot": graph_jpeg.as_posix(),
                })

            combo_results = pd.DataFrame(combo_rows)
            combo_results.to_csv(results_path, index=False)
            rows.extend(combo_rows)
            artifact_rows.append({
                "stage": stage["name"],
                "combo_slug": slug,
                "model": generation_model,
                "embedding_model": embedding_model,
                "index_dir": index_dir.as_posix(),
                "html_plot": graph_html.as_posix(),
                "jpeg_plot": graph_jpeg.as_posix(),
                "results_csv": results_path.as_posix(),
                "question_count": len(combo_results),
                "build_seconds": round(build_seconds, 2),
                "umls_enabled": stage["use_umls"],
                "schema_guided": stage["schema_guided"],
            })
            print(f"{stage['name']} / {slug}: built in {build_seconds:.1f}s")

results = pd.DataFrame(rows)
artifacts = pd.DataFrame(artifact_rows)
artifacts_path = output_root / "artifact_manifest.csv"
results_path = output_root / "evaluation_results.csv"
artifacts.to_csv(artifacts_path, index=False)
results.to_csv(results_path, index=False)
artifacts


## Inline Results

The cells below show the saved artifacts directly in the notebook: manifest tables, static graph previews, optional interactive graph frames, and comparison charts.


In [ ]:
display_columns = [
    "stage",
    "combo_slug",
    "model",
    "embedding_model",
    "build_seconds",
    "question_count",
    "umls_enabled",
    "schema_guided",
    "html_plot",
    "jpeg_plot",
]
artifact_view = artifacts[[column for column in display_columns if column in artifacts.columns]] if not artifacts.empty else artifacts
artifact_view


In [ ]:
max_graph_previews = int(os.environ.get("NOTEBOOK_GRAPH_PREVIEW_LIMIT", "6"))
if artifacts.empty:
    print("No graph artifacts to display yet. Run the pipeline cell first.")
else:
    for row in artifacts.head(max_graph_previews).itertuples(index=False):
        title = f"{row.stage} / {row.combo_slug}"
        print(title)
        jpeg_path = Path(row.jpeg_plot)
        if jpeg_path.exists():
            display(Image(filename=str(jpeg_path), width=900))
        else:
            print(f"Missing JPEG preview: {jpeg_path}")


In [ ]:
interactive_index = int(os.environ.get("NOTEBOOK_INTERACTIVE_GRAPH_INDEX", "0"))
if artifacts.empty:
    print("No interactive graph artifact to display yet.")
else:
    selected = artifacts.iloc[min(interactive_index, len(artifacts) - 1)]
    html_path = Path(selected["html_plot"])
    print(f"Interactive graph: {selected['stage']} / {selected['combo_slug']}")
    if html_path.exists():
        display(IFrame(src=html_path.as_posix(), width="100%", height=720))
    else:
        print(f"Missing HTML graph: {html_path}")


In [ ]:
if artifacts.empty:
    print("No artifact metrics to chart yet.")
else:
    build_chart = artifacts.copy()
    build_chart["label"] = build_chart["stage"] + " | " + build_chart["combo_slug"]
    ax = build_chart.sort_values("build_seconds").plot.barh(
        x="label",
        y="build_seconds",
        figsize=(12, max(4, 0.45 * len(build_chart))),
        legend=False,
        title="Index Build Time by Stage and Model",
    )
    ax.set_xlabel("seconds")
    ax.set_ylabel("")
    display(ax.figure)

if results.empty or "predicted" not in results.columns:
    print("No evaluation predictions to chart yet.")
else:
    scored = results.dropna(subset=["predicted", "gold_answer"]).copy()
    if scored.empty:
        print("Evaluation rows exist, but no scored predictions were available.")
    else:
        scored["correct"] = scored["predicted"].astype(str).str.strip().str.lower() == scored["gold_answer"].astype(str).str.strip().str.lower()
        accuracy = scored.groupby(["stage", "combo_slug"], as_index=False)["correct"].mean()
        accuracy["accuracy"] = accuracy["correct"] * 100
        accuracy["label"] = accuracy["stage"] + " | " + accuracy["combo_slug"]
        ax = accuracy.sort_values("accuracy").plot.barh(
            x="label",
            y="accuracy",
            figsize=(12, max(4, 0.45 * len(accuracy))),
            legend=False,
            title="Evaluation Accuracy by Stage and Model",
        )
        ax.set_xlabel("accuracy (%)")
        ax.set_ylabel("")
        ax.set_xlim(0, 100)
        display(ax.figure)
        accuracy[["stage", "combo_slug", "accuracy"]]


In [ ]:
results
